# AI Enterprise — Zero-Employee Operational Dashboard
**Business:** AI Automation Agency — Supply Chain / SAP Niche  
**Operator:** Sam Leong  
**Status:** Production-Ready  
**Last Updated:** 2026-08-07

## Quick Start
Run all cells to initialize your operational dashboard. This notebook tracks:
- Daily metrics (leads, outreach, replies, revenue)
- Agent performance (uptime, token spend, intervention rate)
- Financial projections (setup fees, retainers, runway)
- Target list management (50-company pipeline)

In [ ]:
# Install dependencies (run once)
# !pip install pandas matplotlib plotly numpy

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML

# Business configuration
CONFIG = {
    "business_name": "AI Automation Agency",
    "niche": "Supply Chain / SAP",
    "operator": "Sam Leong",
    "start_date": "2026-08-07",
    "target_first_dollar_days": 14,
    "currency": "SGD",
    "pricing": {
        "quick_win_setup": 1500,
        "core_setup": 2500,
        "core_monthly": 800,
        "scale_setup": 5000,
        "scale_monthly": 2000,
        "audit": 500
    }
}

---

## 1. DAILY METRICS TRACKER
Track your daily outreach and conversion metrics here.

In [ ]:
# Initialize daily metrics DataFrame
def init_daily_metrics(days=30):
    dates = [datetime.now() - timedelta(days=i) for i in range(days)][::-1]
    data = {
        "date": dates,
        "emails_sent": [0]*days,
        "opens": [0]*days,
        "clicks": [0]*days,
        "replies": [0]*days,
        "diagnostic_booked": [0]*days,
        "diagnostic_completed": [0]*days,
        "deals_won": [0]*days,
        "revenue_setup": [0]*days,
        "revenue_recurring": [0]*days,
        "agent_uptime_pct": [0.0]*days,
        "token_spend_sgd": [0.0]*days,
        "human_interventions": [0]*days
    }
    return pd.DataFrame(data)

# Load or create metrics
try:
    metrics_df = pd.read_csv("/Volumes/Orico e7400 1TB/my-project/ai-enterprise/metrics/daily_metrics.csv", parse_dates=["date"])
except FileNotFoundError:
    metrics_df = init_daily_metrics()

metrics_df.tail(7)

### 1.1 Update Today's Metrics
Run this cell at end of each day to log your metrics.

In [ ]:
today = datetime.now().date()

# Update today's row if it exists, else append
if today in metrics_df["date"].dt.date.values:
    idx = metrics_df[metrics_df["date"].dt.date == today].index[0]
else:
    idx = len(metrics_df)
    metrics_df.loc[idx] = pd.Series(dtype=object)
    metrics_df.loc[idx, "date"] = today

print("Update today's metrics (leave blank to keep existing):")
emails_sent = input(f"Emails sent [{metrics_df.loc[idx, 'emails_sent']}]: ") or metrics_df.loc[idx, 'emails_sent']
opens = input(f"Opens [{metrics_df.loc[idx, 'opens']}]: ") or metrics_df.loc[idx, 'opens']
clicks = input(f"Clicks [{metrics_df.loc[idx, 'clicks']}]: ") or metrics_df.loc[idx, 'clicks']
replies = input(f"Replies [{metrics_df.loc[idx, 'replies']}]: ") or metrics_df.loc[idx, 'replies']
diagnostic_booked = input(f"Diagnostics booked [{metrics_df.loc[idx, 'diagnostic_booked']}]: ") or metrics_df.loc[idx, 'diagnostic_booked']
diagnostic_completed = input(f"Diagnostics completed [{metrics_df.loc[idx, 'diagnostic_completed']}]: ") or metrics_df.loc[idx, 'diagnostic_completed']
deals_won = input(f"Deals won [{metrics_df.loc[idx, 'deals_won']}]: ") or metrics_df.loc[idx, 'deals_won']
revenue_setup = input(f"Setup revenue (SGD) [{metrics_df.loc[idx, 'revenue_setup']}]: ") or metrics_df.loc[idx, 'revenue_setup']
revenue_recurring = input(f"Recurring revenue (SGD) [{metrics_df.loc[idx, 'revenue_recurring']}]: ") or metrics_df.loc[idx, 'revenue_recurring']
agent_uptime = input(f"Agent uptime % [{metrics_df.loc[idx, 'agent_uptime_pct']}]: ") or metrics_df.loc[idx, 'agent_uptime_pct']
token_spend = input(f"Token spend (SGD) [{metrics_df.loc[idx, 'token_spend_sgd']}]: ") or metrics_df.loc[idx, 'token_spend_sgd']
human_interventions = input(f"Human interventions [{metrics_df.loc[idx, 'human_interventions']}]: ") or metrics_df.loc[idx, 'human_interventions']

metrics_df.loc[idx, [
    "emails_sent", "opens", "clicks", "replies",
    "diagnostic_booked", "diagnostic_completed", "deals_won",
    "revenue_setup", "revenue_recurring",
    "agent_uptime_pct", "token_spend_sgd", "human_interventions"
]] = [
    int(emails_sent), int(opens), int(clicks), int(replies),
    int(diagnostic_booked), int(diagnostic_completed), int(deals_won),
    float(revenue_setup), float(revenue_recurring),
    float(agent_uptime), float(token_spend), int(human_interventions)
]

# Save
metrics_df.to_csv("/Volumes/Orico e7400 1TB/my-project/ai-enterprise/metrics/daily_metrics.csv", index=False)
print(f"\nMetrics saved for {today}.")
display(metrics_df.tail(3))

### 1.2 Metrics Dashboard
Visualize your key metrics and conversion funnel.

In [ ]:
# Filter last 14 days
df_vis = metrics_df.tail(14).copy()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_vis["date"], y=df_vis["emails_sent"], mode="lines+markers", name="Emails Sent"))
fig.add_trace(go.Scatter(x=df_vis["date"], y=df_vis["replies"], mode="lines+markers", name="Replies"))
fig.add_trace(go.Scatter(x=df_vis["date"], y=df_vis["deals_won"]*10, mode="lines+markers", name="Deals Won (x10)"))
fig.update_layout(title="14-Day Outreach & Conversion", xaxis_title="Date", yaxis_title="Count")
fig.show()

# Conversion summary
total_sent = metrics_df["emails_sent"].sum()
total_replies = metrics_df["replies"].sum()
total_deals = metrics_df["deals_won"].sum()
total_revenue = metrics_df["revenue_setup"].sum() + metrics_df["revenue_recurring"].sum()

print(f"\n=== CONVERSION SUMMARY (All Time) ===")
print(f"Emails sent: {total_sent}")
print(f"Replies: {total_replies} ({total_replies/total_sent*100:.1f}% reply rate)" if total_sent > 0 else "N/A")
print(f"Deals won: {total_deals}")
print(f"Total revenue: SGD {total_revenue:,.0f}")

---

## 2. FINANCIAL RUNWAY CALCULATOR
Model your cash flow, break-even, and runway based on pricing tiers.

In [ ]:
class FinancialModel:
    def __init__(self):
        self.setup_quick = CONFIG["pricing"]["quick_win_setup"]
        self.setup_core = CONFIG["pricing"]["core_setup"]
        self.monthly_core = CONFIG["pricing"]["core_monthly"]
        self.setup_scale = CONFIG["pricing"]["scale_setup"]
        self.monthly_scale = CONFIG["pricing"]["scale_monthly"]
        self.audit = CONFIG["pricing"]["audit"]
        
        # Monthly costs
        self.costs = {
            "claude_api": 300,
            "n8n_hosting": 50,
            "instantly": 80,
            "notion": 15,
            "slack": 0,
            "monitoring": 20,
            "other": 50
        }
        self.total_monthly_cost = sum(self.costs.values())

    def project_revenue(self, clients_by_tier, months=6):
        """
        clients_by_tier: dict with keys 'quick', 'core', 'scale', 'audits'
        """
        projections = []
        cumulative = 0
        
        for m in range(1, months+1):
            # Setup revenue (new clients this month)
            quick_rev = clients_by_tier.get("quick", 0) * self.setup_quick
            core_rev = clients_by_tier.get("core", 0) * self.setup_core
            scale_rev = clients_by_tier.get("scale", 0) * self.setup_scale
            audit_rev = clients_by_tier.get("audits", 0) * self.audit
            setup_rev = quick_rev + core_rev + scale_rev + audit_rev
            
            # Recurring revenue (all previous clients)
            core_recurring = (m-1) * clients_by_tier.get("core", 0) * self.monthly_core
            scale_recurring = (m-1) * clients_by_tier.get("scale", 0) * self.monthly_scale
            recurring_rev = core_recurring + scale_recurring
            
            total_rev = setup_rev + recurring_rev
            profit = total_rev - self.total_monthly_cost
            cumulative += profit
            
            projections.append({
                "month": m,
                "setup_revenue": setup_rev,
                "recurring_revenue": recurring_rev,
                "total_revenue": total_rev,
                "costs": self.total_monthly_cost,
                "profit": profit,
                "cumulative": cumulative
            })
        
        return pd.DataFrame(projections)

model = FinancialModel()

# Scenario: 1 audit + 1 core client in Month 1, scaling to 3 audits + 2 core by Month 3
scenario = {
    "audits": 2,
    "quick": 1,
    "core": 1,
    "scale": 0
}

projection_df = model.project_revenue(scenario, months=6)
projection_df

In [ ]:
# Visualize financial projection
fig = go.Figure()
fig.add_trace(go.Bar(x=projection_df["month"], y=projection_df["total_revenue"], name="Revenue"))
fig.add_trace(go.Bar(x=projection_df["month"], y=projection_df["costs"], name="Costs"))
fig.add_trace(go.Scatter(x=projection_df["month"], y=projection_df["cumulative"], name="Cumulative Profit", mode="lines+markers"))
fig.update_layout(title="6-Month Financial Projection", xaxis_title="Month", yaxis_title="SGD", barmode="group")
fig.show()

print(f"\n=== FINANCIAL SUMMARY ===")
print(f"Monthly operating cost: SGD {model.total_monthly_cost}")
print(f"Break-even month: Month {projection_df[projection_df['cumulative'] > 0]['month'].min()}")
print(f"6-month cumulative profit: SGD {projection_df['cumulative'].iloc[-1]:,.0f}")

---

## 3. AGENT PERFORMANCE MONITOR
Track agent uptime, token spend, and intervention rates.

In [ ]:
# Agent performance tracker
agents = [
    {"id": "REV-01", "name": "Lead Research", "status": "active", "uptime_pct": 98.5, "daily_tokens": 50000, "cost_sgd": 2.50},
    {"id": "REV-02", "name": "Cold Outreach", "status": "active", "uptime_pct": 99.2, "daily_tokens": 75000, "cost_sgd": 3.75},
    {"id": "REV-03", "name": "Lead Scoring", "status": "active", "uptime_pct": 100.0, "daily_tokens": 25000, "cost_sgd": 1.25},
    {"id": "REV-04", "name": "Proposal Generator", "status": "paused", "uptime_pct": 0.0, "daily_tokens": 0, "cost_sgd": 0.0},
    {"id": "REV-05", "name": "Invoice & Billing", "status": "active", "uptime_pct": 99.8, "daily_tokens": 15000, "cost_sgd": 0.75},
    {"id": "OPS-01", "name": "Workflow Monitor", "status": "active", "uptime_pct": 99.9, "daily_tokens": 5000, "cost_sgd": 0.25},
    {"id": "OPS-02", "name": "QA Auto", "status": "active", "uptime_pct": 97.5, "daily_tokens": 30000, "cost_sgd": 1.50},
    {"id": "OPS-03", "name": "Knowledge Curator", "status": "active", "uptime_pct": 100.0, "daily_tokens": 20000, "cost_sgd": 1.00},
    {"id": "OPS-04", "name": "DevOps & Security", "status": "active", "uptime_pct": 100.0, "daily_tokens": 5000, "cost_sgd": 0.25},
    {"id": "INT-01", "name": "Market Research", "status": "active", "uptime_pct": 100.0, "daily_tokens": 40000, "cost_sgd": 2.00},
    {"id": "INT-02", "name": "Analytics", "status": "active", "uptime_pct": 99.5, "daily_tokens": 10000, "cost_sgd": 0.50},
    {"id": "INT-03", "name": "Forecasting", "status": "active", "uptime_pct": 100.0, "daily_tokens": 15000, "cost_sgd": 0.75},
    {"id": "INT-04", "name": "Content Engine", "status": "paused", "uptime_pct": 0.0, "daily_tokens": 0, "cost_sgd": 0.0},
]

agent_df = pd.DataFrame(agents)
agent_df

In [ ]:
# Agent performance summary
active_agents = agent_df[agent_df["status"] == "active"]
avg_uptime = active_agents["uptime_pct"].mean()
total_daily_cost = active_agents["cost_sgd"].sum()
total_daily_tokens = active_agents["daily_tokens"].sum()

print(f"=== AGENT PERFORMANCE SUMMARY ===")
print(f"Active agents: {len(active_agents)}/{len(agent_df)}")
print(f"Average uptime: {avg_uptime:.1f}%")
print(f"Total daily token cost: SGD {total_daily_cost:.2f}")
print(f"Total daily tokens: {total_daily_tokens:,}")
print(f"Monthly token cost estimate: SGD {total_daily_cost * 30:.2f}")

---

## 4. TARGET LIST MANAGER
Manage your 50-company target list and track outreach status.

In [ ]:
# Target list structure
target_list = [
    {"company": "DB Schenker", "contact": "", "title": "Head of Supply Chain", "country": "Singapore", "industry": "Logistics", "status": "research", "priority": 9},
    {"company": "DHL Supply Chain", "contact": "", "title": "Operations Director", "country": "Singapore", "industry": "3PL", "status": "research", "priority": 8},
    {"company": "Kuehne + Nagel", "contact": "", "title": "SAP MM Lead", "country": "Singapore", "industry": "Freight Forwarding", "status": "research", "priority": 9},
    {"company": "Yusen Logistics", "contact": "", "title": "CFO", "country": "Singapore", "industry": "3PL", "status": "research", "priority": 7},
    {"company": "AGL Logistics", "contact": "", "title": "Supply Chain Manager", "country": "Singapore", "industry": "Logistics", "status": "research", "priority": 7},
]

# Add more targets from Target_List_50_Companies.md
target_df = pd.DataFrame(target_list)
target_df

In [ ]:
# Outreach status summary
status_summary = target_df["status"].value_counts()
print("=== TARGET LIST STATUS ===")
print(status_summary)
print(f"\nTotal targets: {len(target_df)}")
print(f"High priority (>8): {len(target_df[target_df['priority'] > 8])}")
print(f"Research phase: {len(target_df[target_df['status'] == 'research'])}")
print(f"Contacted: {len(target_df[target_df['status'] == 'contacted'])}")
print(f"Responded: {len(target_df[target_df['status'] == 'responded'])}")

---

## 5. DAILY EXECUTION CHECKLIST
Run this every morning to stay on track.

In [ ]:
from datetime import datetime

today = datetime.now()
day_of_week = today.strftime("%A")
launch_date = datetime(2026, 8, 10)  # Day 1 of launch plan
days_since_launch = (today.date() - launch_date.date()).days

print(f"=== DAILY EXECUTION CHECKLIST ===")
print(f"Date: {today.strftime('%Y-%m-%d')} ({day_of_week})")
if days_since_launch >= 0:
    print(f"Launch Day: {days_since_launch + 1}")
else:
    print(f"Days until launch: {abs(days_since_launch)}")

checklist = [
    "[ ] Send 25 personalized outreach emails",
    "[ ] Check reply rate (target >5%)",
    "[ ] Respond to all replies within 4 hours",
    "[ ] Book diagnostic calls via Calendly",
    "[ ] Review agent logs for errors",
    "[ ] Update this metrics notebook",
    "[ ] Check token spend (target <S$500/day)"
]

for item in checklist:
    print(item)

---

## 6. NOTES & ACTION ITEMS
Add your daily notes here.

In [ ]:
print("=== TODAY'S NOTES ===")
notes = input("Enter notes for today (press Enter to skip): ")
if notes:
    with open("/Volumes/Orico e7400 1TB/my-project/ai-enterprise/metrics/daily_notes.txt", "a") as f:
        f.write(f"\n{today.strftime('%Y-%m-%d')}: {notes}")
    print("Notes saved.")